## 🔰PyTorchでニューラルネットワーク基礎 #30 【MLM事前学習編・モデル】

BERTタイプのMLM事前学習で利用するネットワークの部分

### 内容
* Qiitaの記事と連動しています

### 重み共有によるパラーメータ数の削減効果
* Linearのみ：1.6M程度
* 重み共有：1M程度

### ネットワーク構造の基本形
* トークンと位置の埋め込み → Transformer Encoder → Linear (mlm_head) → 単語の予測
* 他のモデルに転用する場合、mlm_headの部分を除いて事前学習済みパラメータを適用、mlm_head部分をタスクに合う形に付け替える。

### これまでとの大きな変更点
1. モデルの設定をクラスファイルで統合した。
   第29回までは、ネットワーク層に数値をそのまま記入していましたが、今回は設定クラスを作成してみました。
    ~~~python
    config = ModelConfig(tokenizer)
    config.d_model 
    ~~~

2. MLMHeadクラス
    * mlm_head部分のBERT風タイプ。論文のコードより実装。weight tyingも追加している。
    * nn.Linear(d_model, vocab_size)という簡易版でもOK。

In [1]:
import torch
import torch.nn as nn
import json
import random
from torch.utils.data import DataLoader, TensorDataset  # ミニバッチの利用 
from tokenizers import Tokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
# サンプルのトークナイザー
tokenizer_filename = "tokenizer/sample_tokenizer_10k.json"
tokenizer = Tokenizer.from_file(tokenizer_filename)

print("特殊トークンID:")
print(f"<unk>: {tokenizer.token_to_id('<unk>')}")
print(f"<bos>: {tokenizer.token_to_id('<bos>')}")
print(f"<eos>: {tokenizer.token_to_id('<eos>')}")
print(f"<pad>: {tokenizer.token_to_id('<pad>')}")
print(f"<mask>: {tokenizer.token_to_id('<mask>')}")
print(f"size: {tokenizer.get_vocab_size()}")

特殊トークンID:
<unk>: 3
<bos>: 1
<eos>: 2
<pad>: 0
<mask>: 4
size: 10000


## ネットワークの構造を決めるクラス
1. モデル構造を指定する定数
2. 特殊トークンなどのID（tokenizerに依存する）
3. 学習時に使う定数

In [3]:
class ModelConfig:
    def __init__(self, tokenizer):
        # モデル構造
        self.vocab_size = tokenizer.get_vocab_size()
        self.d_model = 64
        self.seq_len = 64
        self.nhead = 4
        self.dim_feedforward = 256
        self.num_layers = 6
        self.dropout = 0.1
        
        # 特殊トークンID
        self.pad_token_id = tokenizer.token_to_id("<pad>")
        self.mask_token_id = tokenizer.token_to_id("<mask>")
        self.bos_token_id = tokenizer.token_to_id("<bos>")
        self.eos_token_id = tokenizer.token_to_id("<eos>")
        self.unk_token_id = tokenizer.token_to_id("<unk>")

        # 特殊トークンのセット
        self.special_tokens_set = {
            self.pad_token_id,
            self.mask_token_id,
            self.bos_token_id,
            self.eos_token_id,
            self.unk_token_id,
        }
        
        # 通常トークンのリスト special_tokenを除くトークンのリスト（MLMランダム置換用）
        self.normal_tokens_list = [
            i for i in range(self.vocab_size) 
            if i not in self.special_tokens_set
        ]
        
        # 学習設定 (今回は利用しないけど使うと便利かも)
        self.batch_size = 1024
        self.learning_rate = 0.0001
        self.num_epochs = 100
        self.mask_prob = 0.15
        self.max_grad_norm = 1.0

In [4]:
# MLM用の出力部分
# nn.Linear(d_model, vocab_size)のみからの拡張版

class MLMHead(nn.Module):
    def __init__(self, config: ModelConfig, embedding_weight):
        super().__init__()
        self.fc = nn.Linear(config.d_model, config.d_model)
        self.act = nn.GELU()
        self.ln = nn.LayerNorm(config.d_model)

        self.classification_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.classification_head.weight = embedding_weight  # weight tying 重み共有部分

        self.bias = nn.Parameter(torch.zeros(config.vocab_size))  # バイアス項は学習されるパラメータ

    def forward(self, x):
        x = self.fc(x)
        x = self.act(x)
        x = self.ln(x)
        x = self.classification_head(x) + self.bias # 全結合層と同じ形にするためにバイアス項を足す
        return x

# MLM用のモデル（出力層を変更）
class DNN(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        
        # 埋め込み層
        self.token_embedding = nn.Embedding(
            num_embeddings=config.vocab_size, 
            embedding_dim=config.d_model,
            padding_idx=config.pad_token_id
        )
        self.pos_embedding = nn.Embedding(num_embeddings=config.seq_len, embedding_dim=config.d_model)
        
        self.layer_norm = nn.LayerNorm(config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.nhead,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=config.num_layers)
        
        # MLM用の出力層：BERT風レイヤー　各トークン位置で語彙全体を予測
        self.mlm_head = MLMHead(config, self.token_embedding.weight)    # 重み共有版（こちらのほうが良いはず）
        #self.mlm_head = nn.Linear(config.d_model, config.vocab_size)   # FCのみ
    
    def forward(self, x):
        # マスクの作成
        src_key_padding_mask = (x == self.config.pad_token_id)
        
        # 埋め込み
        tok_emb = self.token_embedding(x)
        pos_emb = self.pos_embedding(torch.arange(x.size(1), device=x.device))
        x = tok_emb + pos_emb.unsqueeze(0)
        
        x = self.layer_norm(x)
        x = self.dropout(x)
        
        # Transformer Encoder
        h = self.transformer_encoder(x, src_key_padding_mask=src_key_padding_mask)
        
        # MLM予測：各位置で語彙全体での確率を出力
        logits = self.mlm_head(h)  # [batch, seq_len, vocab_size]
        
        return logits

In [ ]:
config = ModelConfig(tokenizer)   # tokenizerを指定して設定
model = DNN(config)               # configを使ってモデルを作成
model.to(device)

DNN(
  (token_embedding): Embedding(10000, 64, padding_idx=0)
  (pos_embedding): Embedding(64, 64)
  (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (mlm_head): MLMHead(
    (fc): Linear(in_features=64, out_features=64, bias=

## 学習パラメータの数

In [6]:
def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"学習されるパラメータ数: {count_trainable_parameters(model):,}")

学習されるパラメータ数: 958,416
